# Stolen model detection

This notebook scores 360 suspect CIFAR-100 ResNet-18 models against the target model. It combines parameter-level evidence, output-distribution similarity, representation similarity, augmentation response, boundary probes, and gradient-based fingerprints. The final submission uses rank-normalized scores for each suspect model.


In [ ]:
import subprocess
import sys

required_packages = ["safetensors", "huggingface_hub", "tqdm"]
for package in required_packages:
    try:
        __import__(package.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


In [ ]:
import gc
import json
import math
import os
import random
import re
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet18
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm

IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    pass

print("Environment:", "Kaggle" if IN_KAGGLE else ("Colab" if IN_COLAB else "Local"))


In [ ]:
SEED = 1337
BATCH_SIZE = 256
NUM_WORKERS = 2
N_MEMBER = 2000
N_TEST = 2000
N_AUG = 2000
N_JAC = 64

# Target training augmentation disclosed in the assignment.
MEAN = (0.5071, 0.4867, 0.4408)
STD = (0.2675, 0.2565, 0.2761)
REFLECT_PADDING = 4
BIAS_X = 0.5
BIAS_Y = -0.25
JITTER = 0.25

REPO_ID = "SprintML/tml26_task2"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

if IN_KAGGLE:
    ROOT = Path("/kaggle/working/tml_12")
    DATA_ROOT = Path("/kaggle/working/data")
    HF_CACHE_DIR = Path("/kaggle/working/hf_cache")
    BASE_DIR = ROOT / "hf_repo"
elif IN_COLAB:
    ROOT = Path("/content/tml_v12")
    DATA_ROOT = Path("/content/data")
    HF_CACHE_DIR = Path("/content/hf_cache")
    BASE_DIR = ROOT / "hf_repo"
else:
    ROOT = Path("./tml_v12")
    DATA_ROOT = Path("./data")
    HF_CACHE_DIR = Path("./hf_cache")
    BASE_DIR = ROOT / "hf_repo"

ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")

FEATURE_PATH = ROOT / "features_v12.csv"
SUBMISSION_DIR = ROOT / "submissions"
SUBMISSION_DIR.mkdir(exist_ok=True)

print("Device:", DEVICE)
print("Root:", ROOT)


In [ ]:
snapshot_download(
    repo_id=REPO_ID,
    repo_type="model",
    local_dir=str(BASE_DIR),
    local_dir_use_symlinks=False,
    resume_download=True,
)

TARGET_PATH = BASE_DIR / "target_model" / "weights.safetensors"
TRAIN_IDX_PATH = BASE_DIR / "target_model" / "train_main_idx.json"
SUSPECT_DIR = BASE_DIR / "suspect_models"
suspect_paths = sorted(SUSPECT_DIR.glob("suspect_*.safetensors"))

if not TARGET_PATH.exists():
    candidates = list((BASE_DIR / "target_model").glob("*.safetensors"))
    if not candidates:
        raise FileNotFoundError("No target weights found")
    TARGET_PATH = candidates[0]

assert len(suspect_paths) == 360, f"Expected 360 suspects, got {len(suspect_paths)}"
print("Target:", TARGET_PATH.name)
print("Suspects:", len(suspect_paths))


In [ ]:
def make_model():
    model = resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 100)
    return model


def load_state(path):
    return load_file(str(path), device="cpu")


def load_model(state_dict):
    model = make_model()
    model.load_state_dict(state_dict, strict=True)
    return model.to(DEVICE).eval()


def get_id(path):
    return int(re.search(r"(\d+)$", path.stem).group(1))


target_state = load_state(TARGET_PATH)
target_model = load_model(target_state)
assert sorted(get_id(path) for path in suspect_paths) == list(range(360))
print("Target parameters:", len(target_state))


In [ ]:
def disclosed_biased_crop(image, item_seed):
    rng = np.random.default_rng(SEED + int(item_seed))
    padded = TF.pad(image, [REFLECT_PADDING] * 4, padding_mode="reflect")
    jitter_x = rng.uniform(-JITTER, JITTER) * REFLECT_PADDING
    jitter_y = rng.uniform(-JITTER, JITTER) * REFLECT_PADDING
    left = int(np.clip(round(REFLECT_PADDING + BIAS_X * REFLECT_PADDING + jitter_x), 0, 2 * REFLECT_PADDING))
    top = int(np.clip(round(REFLECT_PADDING + BIAS_Y * REFLECT_PADDING + jitter_y), 0, 2 * REFLECT_PADDING))
    return TF.crop(padded, top=top, left=left, height=32, width=32)


def disclosed_aug_view(image, item_seed):
    cropped = disclosed_biased_crop(image, item_seed)
    rng = np.random.default_rng(SEED + 10_000 + int(item_seed))
    return TF.hflip(cropped) if rng.random() < 0.5 else cropped


class CleanSubset(Dataset):
    def __init__(self, base, indices):
        self.base = base
        self.indices = list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        image, label = self.base[idx]
        return TF.normalize(TF.to_tensor(image), MEAN, STD), int(label)


class AugSubset(Dataset):
    def __init__(self, base, indices, seed_offset=0):
        self.base = base
        self.indices = list(indices)
        self.seed_offset = seed_offset

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        image, label = self.base[idx]
        image = disclosed_aug_view(image, idx + self.seed_offset)
        return TF.normalize(TF.to_tensor(image), MEAN, STD), int(label)


class TensorDS(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], int(self.y[i])


raw_train = datasets.CIFAR100(root=str(DATA_ROOT), train=True, download=True, transform=None)
raw_test = datasets.CIFAR100(root=str(DATA_ROOT), train=False, download=True, transform=None)

with open(TRAIN_IDX_PATH) as f:
    target_train_idx = json.load(f)

rng = np.random.default_rng(SEED)
member_idx = rng.choice(target_train_idx, size=N_MEMBER, replace=False).tolist()
test_idx = rng.choice(np.arange(len(raw_test)), size=N_TEST, replace=False).tolist()
aug_idx = rng.choice(target_train_idx, size=N_AUG, replace=False).tolist()

base_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
train_tf = datasets.CIFAR100(root=str(DATA_ROOT), train=True, download=False, transform=base_tf)
cutmix_loader = DataLoader(Subset(train_tf, member_idx), batch_size=256, shuffle=False, num_workers=NUM_WORKERS)
cm_x = torch.cat([x for x, _ in cutmix_loader], dim=0)


def make_cutmix(x, seed=42, lam=0.5):
    generator = torch.Generator().manual_seed(seed)
    n, _, h, w = x.shape
    perm = torch.randperm(n, generator=generator)
    cut_ratio = math.sqrt(1.0 - lam)
    cut_w, cut_h = int(w * cut_ratio), int(h * cut_ratio)
    cx, cy = w // 2, h // 2
    x1, x2 = max(cx - cut_w // 2, 0), min(cx + cut_w // 2, w)
    y1, y2 = max(cy - cut_h // 2, 0), min(cy + cut_h // 2, h)
    out = x.clone()
    out[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
    return out


cutmix_x = make_cutmix(cm_x, seed=42, lam=0.5)
cutmix_y = torch.zeros(len(cutmix_x), dtype=torch.long)

test_tf = datasets.CIFAR100(root=str(DATA_ROOT), train=False, download=False, transform=base_tf)
jac_loader = DataLoader(Subset(test_tf, test_idx[:N_JAC]), batch_size=N_JAC, shuffle=False, num_workers=0)
jac_x = next(iter(jac_loader))[0]

loaders = {
    "member": DataLoader(CleanSubset(raw_train, member_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
    "test": DataLoader(CleanSubset(raw_test, test_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
    "cutmix": DataLoader(TensorDS(cutmix_x, cutmix_y), batch_size=BATCH_SIZE, shuffle=False, num_workers=0),
    "member_aug": DataLoader(AugSubset(raw_train, aug_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
    "member_clean": DataLoader(CleanSubset(raw_train, aug_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
}

for name, loader in loaders.items():
    print(f"{name:>12}: {len(loader.dataset)}")
print("jacobian probe:", tuple(jac_x.shape))


In [ ]:
# Basic similarity utilities
def cos_np(a, b):
    a, b = np.asarray(a, np.float64).ravel(), np.asarray(b, np.float64).ravel()
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-12))

def spearman(a, b):
    a, b = np.asarray(a, np.float64).ravel(), np.asarray(b, np.float64).ravel()
    if a.size < 2: return 0.0
    ra = a.argsort().argsort().astype(np.float64); ra -= ra.mean()
    rb = b.argsort().argsort().astype(np.float64); rb -= rb.mean()
    return float(np.dot(ra,rb)/(np.linalg.norm(ra)*np.linalg.norm(rb)+1e-12))

# Model outputs used by several feature groups
@torch.no_grad()
def collect_outputs(model, loaders_dict):
    out = {}; model.eval()
    for name, loader in loaders_dict.items():
        ll, fl, yl = [], [], []
        for batch in loader:
            x = batch[0].to(DEVICE, non_blocking=True); y = batch[1]
            f = model.conv1(x); f = model.bn1(f); f = model.relu(f); f = model.maxpool(f)
            f = model.layer1(f); f = model.layer2(f); f = model.layer3(f); f = model.layer4(f)
            feat = torch.flatten(model.avgpool(f), 1); logits = model.fc(feat)
            ll.append(logits.cpu().float()); fl.append(feat.cpu().float())
            yl.append(y if isinstance(y, torch.Tensor) else torch.tensor(y))
        logits = torch.cat(ll).numpy(); feat = torch.cat(fl).numpy(); labels = torch.cat(yl).numpy()
        probs  = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
        top2   = np.sort(logits, axis=1)[:, -2:]; margin = top2[:,1] - top2[:,0]
        out[name] = dict(logits=logits.astype(np.float32), feat=feat.astype(np.float32),
                         probs=probs.astype(np.float32), pred=logits.argmax(axis=1).astype(np.int64),
                         margin=margin.astype(np.float32), labels=labels.astype(np.int64))
    return out

# Sample correlation similarity
def sample_corr_matrix(X):
    X = X.astype(np.float64); X -= X.mean(axis=1,keepdims=True)
    X /= np.linalg.norm(X, axis=1, keepdims=True)+1e-12; return X @ X.T

def sac_score(tp, sp):
    Ct, Cs = sample_corr_matrix(tp), sample_corr_matrix(sp)
    iu = np.triu_indices(Ct.shape[0], k=1); vt, vs = Ct[iu], Cs[iu]
    return float(np.dot(vt,vs)/(np.linalg.norm(vt)*np.linalg.norm(vs)+1e-12))

# Linear CKA on penultimate features
def linear_cka(X, Y):
    X, Y = X.astype(np.float64), Y.astype(np.float64)
    X -= X.mean(axis=0,keepdims=True); Y -= Y.mean(axis=0,keepdims=True)
    xy = np.linalg.norm(X.T@Y,"fro")**2
    return float(xy/(np.linalg.norm(X.T@X,"fro")*np.linalg.norm(Y.T@Y,"fro")+1e-12))

# Input-gradient similarity
def input_grad(model, x, ref):
    x = x.detach().clone().requires_grad_(True)
    model(x).gather(1, ref.view(-1,1)).sum().backward()
    return x.grad.detach()

def jacobian_cosine(t_model, s_model, probe_x, bs=32):
    t_model.eval(); s_model.eval(); tot = 0.0; n = 0
    for i in range(0, probe_x.size(0), bs):
        xb = probe_x[i:i+bs].to(DEVICE)
        with torch.no_grad(): ref = t_model(xb).argmax(dim=1)
        gt = input_grad(t_model, xb, ref).flatten(1)
        gs = input_grad(s_model, xb, ref).flatten(1)
        tot += float(F.cosine_similarity(gt, gs, dim=1).sum().item()); n += xb.size(0)
    return tot / max(n, 1)

# Batch-normalization fingerprints
def bn_stats_features(t_state, s_state):
    rmt, rms, rvt, rvs, aft, afs = [], [], [], [], [], []
    for k, t in t_state.items():
        if not isinstance(t, torch.Tensor): continue
        s = s_state[k]
        if   "running_mean" in k: rmt.append(t.float().flatten()); rms.append(s.float().flatten())
        elif "running_var"  in k: rvt.append(t.float().flatten()); rvs.append(s.float().flatten())
        elif ("bn" in k or "downsample.1" in k) and ("weight" in k or "bias" in k) \
             and "running_" not in k and "num_batches" not in k:
            aft.append(t.float().flatten()); afs.append(s.float().flatten())
    out = {}
    if rmt:
        tm, sm = torch.cat(rmt).numpy(), torch.cat(rms).numpy()
        out["bn_runmean_cos"]    = cos_np(tm, sm)
        out["bn_runmean_l1_neg"] = -float(np.mean(np.abs(tm - sm)))
    if rvt:
        tv, sv = torch.cat(rvt).numpy(), torch.cat(rvs).numpy()
        out["bn_runvar_logcos"]  = cos_np(np.log1p(tv), np.log1p(sv))
    if aft:
        ta, sa = torch.cat(aft).numpy(), torch.cat(afs).numpy()
        out["bn_affine_cosine"]  = cos_np(ta, sa)
    return out

# Response to the disclosed training augmentation
def centered_cosine_np(a, b):
    a = a - a.mean(axis=1, keepdims=True); b = b - b.mean(axis=1, keepdims=True)
    na = np.linalg.norm(a, axis=1, keepdims=True)+1e-12
    nb = np.linalg.norm(b, axis=1, keepdims=True)+1e-12
    return (a/na * b/nb).sum(axis=1)

def class_delta_stats(delta_cos, t_pred, min_count=5):
    class_means = [delta_cos[t_pred==c].mean() for c in np.unique(t_pred) if (t_pred==c).sum()>=min_count]
    if not class_means: return {"p10": np.nan, "p50": np.nan, "iqr_neg": np.nan}
    arr = np.asarray(class_means)
    p10,p25,p50,p75 = np.quantile(arr,[.10,.25,.50,.75])
    return {"p10":float(p10), "p50":float(p50), "iqr_neg":float(-(p75-p25))}

def augmentation_features(t_clean, t_aug, s_clean, s_aug):
    t_delta   = t_aug["logits"].astype(np.float64)   - t_clean["logits"].astype(np.float64)
    s_delta   = s_aug["logits"].astype(np.float64)   - s_clean["logits"].astype(np.float64)
    delta_cos = centered_cosine_np(t_delta, s_delta)
    cs        = class_delta_stats(delta_cos, t_clean["pred"])
    t_ap, s_ap = t_aug["logits"].argmax(axis=1), s_aug["logits"].argmax(axis=1)
    t_chg = t_clean["pred"] != t_ap; s_chg = s_clean["pred"] != s_ap
    return {
        "aug_delta_logit_cos":     float(delta_cos.mean()),
        "aug_class_delta_p10":     cs["p10"],
        "aug_class_delta_p50":     cs["p50"],
        "aug_class_delta_iqr_neg": cs["iqr_neg"],
        "aug_change_agreement":    float((t_chg == s_chg).mean()),
        "aug_label_agreement":     float((t_ap == s_ap).mean()),
    }


# Decision Distance Vector similarity
def ddv_cosine(t_probs, s_probs, n_pairs=2000, seed=42):
    """Compare pairwise distances between output distributions."""
    rng = np.random.default_rng(seed)
    n = len(t_probs)
    idx_a = rng.integers(0, n, size=n_pairs)
    idx_b = rng.integers(0, n, size=n_pairs)
    t_ddv = np.linalg.norm(t_probs[idx_a] - t_probs[idx_b], axis=1)
    s_ddv = np.linalg.norm(s_probs[idx_a] - s_probs[idx_b], axis=1)
    denom = np.linalg.norm(t_ddv) * np.linalg.norm(s_ddv) + 1e-12
    return float(np.dot(t_ddv, s_ddv) / denom)


# SAC on target mistakes
def sac_score_wrong(t_probs, s_probs, t_pred, t_labels, min_wrong=10):
    """SAC restricted to target mistakes, where errors are more model-specific."""
    valid = t_labels >= 0
    if not valid.any():
        return np.nan
    wrong = valid & (t_pred != t_labels)
    if int(wrong.sum()) < min_wrong:
        return np.nan
    return sac_score(t_probs[wrong], s_probs[wrong])


# SAC on high-confidence target examples
def sac_score_highconf(t_probs, s_probs, percentile=80):
    """SAC on high-confidence target predictions."""
    conf = t_probs.max(axis=1)
    threshold = np.percentile(conf, percentile)
    mask = conf >= threshold
    if mask.sum() < 10:
        return np.nan
    return sac_score(t_probs[mask], s_probs[mask])


# Target boundary probes
def generate_boundary_probes(model, images, steps=10, step_size=1.0/255):
    """Move images toward the target decision boundary."""
    model.eval()
    x = images.clone().to(DEVICE)
    for _ in range(steps):
        x = x.detach().requires_grad_(True)
        logits = model(x)
        probs = logits.softmax(dim=1)
        top2_vals = probs.topk(2, dim=1).values
        # Smaller top-1/top-2 margins place probes near the target boundary.
        loss = (top2_vals[:, 0] - top2_vals[:, 1]).mean()
        loss.backward()
        x = (x - step_size * x.grad.sign()).detach()
    return x.detach().cpu()


# Per-layer representation cosine
@torch.no_grad()
def collect_layer_outputs(model, images_tensor, batch_size=256):
    """Collect GAP-pooled activations from ResNet layer1 through layer4."""
    model.eval()
    layer_chunks = {f"layer{i}": [] for i in range(1, 5)}
    handles = []

    for i in range(1, 5):
        layer = getattr(model, f"layer{i}")
        def make_hook(name):
            def hook(m, inp, out):
                pooled = F.adaptive_avg_pool2d(out.detach(), 1).flatten(1).cpu()
                layer_chunks[name].append(pooled)
            return hook
        handles.append(layer.register_forward_hook(make_hook(f"layer{i}")))

    for start in range(0, len(images_tensor), batch_size):
        model(images_tensor[start:start+batch_size].to(DEVICE, non_blocking=True))

    for h in handles:
        h.remove()

    return {name: torch.cat(chunks, dim=0).numpy().astype(np.float32)
            for name, chunks in layer_chunks.items()}


def layer_cosine_features(t_layers, s_layers):
    """Average per-example cosine similarity at each ResNet stage."""
    out = {}
    for name in ["layer1", "layer2", "layer3", "layer4"]:
        t, s = t_layers[name].astype(np.float64), s_layers[name].astype(np.float64)
        num = (t * s).sum(axis=1)
        den = np.linalg.norm(t, axis=1) * np.linalg.norm(s, axis=1) + 1e-12
        out[f"{name}_cos"] = float((num / den).mean())
    vals = list(out.values())
    out["layer_cos_mean"] = float(np.mean(vals))
    out["layer_cos_min"]  = float(np.min(vals))
    return out


print("Feature functions ready")


In [ ]:
target_out = collect_outputs(target_model, loaders)
for name, values in target_out.items():
    print(name, values["logits"].shape)

print("Generating boundary probes")
member_images = cm_x[:200]
boundary_x = generate_boundary_probes(target_model, member_images, steps=10)
print("Boundary probes:", tuple(boundary_x.shape))

print("Collecting target layer activations")
target_layer_feats = collect_layer_outputs(target_model, cm_x[:500])
print("Layer features:", {name: value.shape for name, value in target_layer_feats.items()})


In [ ]:
rows, done_ids = [], set()
if FEATURE_PATH.exists():
    old = pd.read_csv(FEATURE_PATH)
    rows = old.to_dict("records")
    done_ids = set(old["id"].astype(int).tolist())
    print("Resuming from cached features:", len(done_ids))

for path in tqdm(suspect_paths, desc="Scoring"):
    sid = get_id(path)
    if sid in done_ids: continue

    suspect_state = load_state(path)
    suspect_model = load_model(suspect_state)
    suspect_out   = collect_outputs(suspect_model, loaders)

    row = {"id": sid}

    # Batch-normalization statistics and affine parameters
    row.update(bn_stats_features(target_state, suspect_state))

    # Output, representation, margin, and prediction-agreement features
    for name in ["member", "test", "cutmix"]:
        row[f"{name}_sac"]             = sac_score(target_out[name]["probs"],  suspect_out[name]["probs"])
        row[f"{name}_cka"]             = linear_cka(target_out[name]["feat"],  suspect_out[name]["feat"])
        row[f"{name}_margin_spearman"] = spearman(target_out[name]["margin"],  suspect_out[name]["margin"])
        row[f"{name}_pred_agree"]      = float((target_out[name]["pred"] == suspect_out[name]["pred"]).mean())

    # Response to the disclosed training augmentation
    row.update(augmentation_features(
        target_out["member_clean"], target_out["member_aug"],
        suspect_out["member_clean"], suspect_out["member_aug"],
    ))

    # Pairwise output-distance geometry
    for name in ["member", "test", "cutmix"]:
        row[f"{name}_ddv"] = ddv_cosine(target_out[name]["probs"],
                                         suspect_out[name]["probs"], n_pairs=2000)

    # Target mistakes are usually more distinctive than easy correct examples.
    for name in ["member", "test"]:
        row[f"{name}_sac_w"] = sac_score_wrong(
            target_out[name]["probs"], suspect_out[name]["probs"],
            target_out[name]["pred"],  target_out[name]["labels"]
        )

    # High-confidence agreement captures stable target behavior.
    for name in ["member", "test"]:
        row[f"{name}_sac_hc"] = sac_score_highconf(
            target_out[name]["probs"], suspect_out[name]["probs"]
        )

    # Boundary probes emphasize decision-surface similarity.
    _b_t = collect_outputs(target_model, {"boundary": DataLoader(
        TensorDS(boundary_x, torch.zeros(len(boundary_x), dtype=torch.long)),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)})
    _b_s = collect_outputs(suspect_model, {"boundary": DataLoader(
        TensorDS(boundary_x, torch.zeros(len(boundary_x), dtype=torch.long)),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)})
    row["boundary_pred_agree"] = float((_b_t["boundary"]["pred"] == _b_s["boundary"]["pred"]).mean())
    row["boundary_sac"]        = sac_score(_b_t["boundary"]["probs"], _b_s["boundary"]["probs"])

    # Layer-wise representation similarity
    sus_layer_feats = collect_layer_outputs(suspect_model, cm_x[:500])
    row.update(layer_cosine_features(target_layer_feats, sus_layer_feats))

    # Input-gradient similarity
    row["jacobian_cos"] = jacobian_cosine(target_model, suspect_model, jac_x)

    rows.append(row)
    pd.DataFrame(rows).sort_values("id").to_csv(FEATURE_PATH, index=False)

    del suspect_model, suspect_state, suspect_out
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

features = pd.read_csv(FEATURE_PATH).sort_values("id").reset_index(drop=True)
print("Feature table:", features.shape)
features.head()


In [ ]:
# Scoring
def robust_z(x):
    x = np.asarray(x, np.float64)
    finite = np.isfinite(x)
    if not finite.any():
        return np.zeros_like(x, dtype=np.float64)
    fill = np.nanmedian(x[finite])
    x = np.where(finite, x, fill)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if not np.isfinite(mad) or mad < 1e-12:
        return np.zeros_like(x, dtype=np.float64)
    z = (x - med) / (1.4826 * mad)
    return np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)

def to_rank_unit(s):
    values = pd.Series(s, dtype="float64").replace([np.inf, -np.inf], np.nan)
    if values.notna().sum() == 0:
        return np.full(len(values), 0.5, dtype=float)
    values = values.fillna(values.median())
    if values.nunique(dropna=True) <= 1:
        return np.full(len(values), 0.5, dtype=float)
    return values.rank(method="average", pct=True).values

def zframe(df, cols):
    z = pd.DataFrame(index=df.index)
    for c in [c for c in cols if c in df.columns]:
        v = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
        z[c] = robust_z(v.values)
    return z

# Feature groups
bn_cols     = [c for c in ["bn_runmean_cos","bn_runmean_l1_neg","bn_runvar_logcos","bn_affine_cosine"] if c in features.columns]
sac_cols    = [c for c in features.columns if c.endswith("_sac")]
cka_cols    = [c for c in features.columns if c.endswith("_cka")]
margin_cols = [c for c in features.columns if c.endswith("_margin_spearman")]
pred_cols   = [c for c in features.columns if c.endswith("_pred_agree")]
jac_cols    = ["jacobian_cos"] if "jacobian_cos" in features.columns else []
aug_cols    = [c for c in features.columns if c.startswith("aug_")]
ddv_cols    = [c for c in features.columns if c.endswith("_ddv")]
sacw_cols   = [c for c in features.columns if c.endswith("_sac_w")]
sachc_cols  = [c for c in features.columns if c.endswith("_sac_hc")]
bnd_cols    = [c for c in features.columns if c.startswith("boundary_")]
layer_cols  = [c for c in features.columns if c.endswith("_cos") and "layer" in c]

print("Group sizes:")
for name, cols in [("BN",bn_cols),("SAC",sac_cols),("CKA",cka_cols),
                   ("Margin",margin_cols),("Pred",pred_cols),("Jac",jac_cols),
                   ("Aug",aug_cols),("DDV",ddv_cols),("SAC-w",sacw_cols),
                   ("SAC-hc",sachc_cols),("Boundary",bnd_cols),("Layer",layer_cols)]:
    print(f"  {name:>8}: {len(cols)}  {cols}")

z_bn = zframe(features, bn_cols); z_sac = zframe(features, sac_cols)
z_cka = zframe(features, cka_cols); z_marg = zframe(features, margin_cols)
z_pred = zframe(features, pred_cols); z_jac = zframe(features, jac_cols)
z_aug   = zframe(features, aug_cols)
z_ddv   = zframe(features, ddv_cols)
z_sacw  = zframe(features, sacw_cols)
z_sachc = zframe(features, sachc_cols)
z_bnd   = zframe(features, bnd_cols)
z_layer = zframe(features, layer_cols)
z_all   = pd.concat([z_bn, z_sac, z_cka, z_marg, z_pred, z_jac,
                     z_aug, z_ddv, z_sacw, z_sachc, z_bnd, z_layer], axis=1)
z_all_values = np.nan_to_num(z_all.to_numpy(dtype=float), nan=0.0, posinf=0.0, neginf=0.0)

def mz(z):
    if z.empty:
        return np.zeros(len(features))
    return np.nan_to_num(z.mean(axis=1).values, nan=0.0, posinf=0.0, neginf=0.0)

def row_topk_mean(values, k):
    if values.shape[1] == 0:
        return np.zeros(values.shape[0])
    k = min(k, values.shape[1])
    return np.sort(values, axis=1)[:, -k:].mean(axis=1)
bn_s, sac_s, cka_s, marg_s, jac_s, aug_s = (mz(z) for z in (z_bn,z_sac,z_cka,z_marg,z_jac,z_aug))
ddv_s, sacw_s, sachc_s, bnd_s, layer_s = (mz(z) for z in (z_ddv,z_sacw,z_sachc,z_bnd,z_layer))

variants = {
    # Single-group diagnostics
    "bn_only":     bn_s,
    "sac_only":    sac_s,
    "cka_only":    cka_s,
    "jacobian":    jac_s,
    "aug_only":    aug_s,
    # Global aggregators
    "max":         z_all_values.max(axis=1),
    "top3_mean":   row_topk_mean(z_all_values, 3),
    "top5_mean":   row_topk_mean(z_all_values, 5),
    # Weighted variants kept for comparison
    "ensemble_v2": 0.45*bn_s + 0.25*sac_s + 0.20*cka_s + 0.10*jac_s,
    "ensemble_v5": 0.30*bn_s + 0.20*sac_s + 0.15*cka_s + 0.08*jac_s + 0.20*aug_s + 0.07*marg_s,
    "bn_or_func":  np.maximum(bn_s, np.max(np.stack([sac_s,cka_s,jac_s,aug_s]),axis=0)),
    # Additional diagnostics
    "ddv_only":    ddv_s,
    "sacw_only":   sacw_s,
    "sachc_only":  sachc_s,
    "boundary":    bnd_s,
    "layer_only":  layer_s,
    # Weighted ensemble over all feature families
    "ensemble_v11": (0.25*bn_s + 0.18*sac_s + 0.12*cka_s + 0.08*jac_s
                    + 0.15*aug_s + 0.08*ddv_s + 0.05*sacw_s
                    + 0.04*sachc_s + 0.03*bnd_s + 0.02*layer_s),
}

for name, raw in variants.items():
    sub = pd.DataFrame({"id": features["id"].astype(int).values, "score": to_rank_unit(raw)})
    sub.to_csv(SUBMISSION_DIR / f"submission_{name}.csv", index=False)
    if IN_KAGGLE:
        sub.to_csv(f"/kaggle/working/submission_{name}.csv", index=False)

# Default submission variant
import shutil
shutil.copy(SUBMISSION_DIR / "submission_max.csv", ROOT / "submission.csv")
if IN_KAGGLE:
    shutil.copy(SUBMISSION_DIR / "submission_max.csv", "/kaggle/working/submission.csv")

diag = pd.DataFrame({"id": features["id"].astype(int).values})
for name, raw in variants.items():
    diag[name] = to_rank_unit(raw)
print("\nTop 15 by max:")
print(diag.sort_values("max", ascending=False).head(15).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

def plot_if_finite(ax, frame, label, bins=30):
    if label not in frame.columns:
        return
    values = pd.to_numeric(frame[label], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if values.empty:
        print(f"Skipping histogram for {label}: no finite values")
        return
    ax.hist(values, bins=bins, alpha=0.5, label=label)

for lbl in ["max", "ensemble_v11", "ensemble_v5", "ensemble_v2", "top3_mean"]:
    plot_if_finite(axes[0], diag, lbl)
axes[0].set_title("Combined variants")
axes[0].set_xlabel("Score")
axes[0].legend()
for lbl in ["bn_only", "sac_only", "ddv_only", "sacw_only", "sachc_only", "boundary", "layer_only", "aug_only"]:
    plot_if_finite(axes[1], diag, lbl)
axes[1].set_title("Single-group diagnostics")
axes[1].set_xlabel("Score")
axes[1].legend()
plt.tight_layout()
plt.show()


Then the submission.csv should be submitted.